In [2]:
import pandas as pd

# Загружаем данные и фиксируем типы ID
clients = pd.read_excel("id_clients.xlsx",dtype={"ID клиентов": "string"})
db = pd.read_excel("data_base.xlsx", dtype={"code": "string"})

In [6]:
# Создаем датафрейм df из clients и db с помощью inner join по id клиентов + если в clients будут дубликаты, то выдаст ошибку
df = db.merge(
    clients,
    left_on="code",
    right_on="ID клиентов",
    how="inner",
    validate="many_to_one"
)

# Оставляем только Малый и Микро сегменты бизнеса и месячную периодичность
df = df[df["segmca"].isin(["Малые", "Микро"]) & df["period_type"].eq("M")].copy()

# Заполняем пропуски в названии продукта значением "Без продукта"
df["product"] = df["product"].fillna("Без продукта")

# Приводим дату к месяцу
df['date_report'] = pd.to_datetime(df['date_report'])
df['month'] = df['date_report'].dt.to_period('M').astype(str)

# Группируем по клиентам
df = (
    df.groupby(
        ["code", "pl_type", "product", "month"],
        as_index=False
    )["c_sum"]
    .sum()
)

# Рассчитываем среднюю доходность клиентов малого и микро бизнеса по типу PL, продукту и месяцам
result = pd.pivot_table(
    df,
    values="c_sum",
    index=["pl_type", "product"],
    columns="month",
    aggfunc="mean"
).reset_index()

# Убираем имя оси столбцов
result.columns.name = None

# Сохраняем результат в xlsx файл
result.to_excel("result.xlsx", index=False)

In [7]:
display(result)

,pl_type,product,2026-01,2026-02,2026-03,2026-04,2026-05
0,ACTIVE,Полученные штрафы по кредитам ЮЛ,16.760000,0.000000,0.000000,NaN,NaN
1,NKD,Безналичные операции корпоративных клиентов,NaN,832.706667,718.966667,0.000000,NaN
2,NKD,Ведение счетов,NaN,245.900000,NaN,NaN,NaN
3,NKD,Доходы банка-эмитента от операций в сторонней ...,NaN,NaN,23.930000,NaN,NaN
4,NKD,Интерчейндж,NaN,NaN,NaN,-14181.077300,NaN
5,NKD,Обслуживание по тарифным планам,NaN,5061.000000,1290.000000,1790.000000,NaN
6,NKD,Терр. распределение по ЮЛ,NaN,NaN,-832.563300,NaN,NaN
7,NKD,Терр. распределение по карте,NaN,42.773600,991.938400,97.055936,NaN
8,NKD,Торговое финансирование и документарные операции,NaN,13621.940000,NaN,NaN,NaN
9,OTHER,Прочие взаиморасчеты,-2147.938365,-433.383552,-1608.961208,-450.123693,NaN
